# Notebook 2 — Are forecast intervals trustworthy in production?

**Headline question for a dispatch desk:** Can we trust Week 3 P10–P90 bands as an **80% envelope** for regulating reserve?

This notebook answers that with **conformalized quantile regression (CQR)** on the Week 3 quantile LightGBM stack:

1. **Question** — raw quantile coverage vs nominal
2. **Method** — split conformal + CQR with chronological calibration
3. **Fit** — quantile LightGBM (P05/P10/P50/P90/P95)
4. **Raw coverage** — empirical P10–P90 and P05–P95 on a held-out test block
5. **MAPIE CQR** — conformal correction at 80% and 90%
6. **Numbers** — coverage table + fan plot
7. **Dispatch read** — under-calibrated bands under-procure reserve
8. **Caveat** — one held-out slice; rolling backtest is the next increment

See also: [`docs/conformal_mental_model.md`](../docs/conformal_mental_model.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.conformal.cqr import run_conformal_cqr
from src.conformal.quantile_lgbm import WEEK3_MODEL_PARAMS, QuantileLGBM
from src.conformal.simulate import WindSimulationConfig, feature_columns, simulate_wind_forecast
from src.conformal.split import chronological_conformal_split

## 1. Question

Week 3 trains separate LightGBM models at **P10, P50, P90** with pinball loss. Default models often **under-cover** (~59% empirical vs 80% nominal on June 2019 CV).

A TSO sizing reserve from P10–P90 needs **empirical coverage**, not just a label. Conformal prediction widens under-covering bands until marginal coverage matches nominal — with no distributional assumption.

**Fine print:** guarantees need approximate **exchangeability**. We use a **chronological** train → calibration → test split (24 h gap), not a random shuffle.

## 2. Data and split

Synthetic DE-style day-ahead wind (~120 days) with heteroskedastic tails — the Week 3 under-coverage story in miniature. If a sibling `wind-quantile-forecast/data/processed/day_ahead_wind.parquet` exists locally, swap it in here.

In [ ]:
wind_parquet = (
    Path("..") / ".." / "wind-quantile-forecast" / "data" / "processed" / "day_ahead_wind.parquet"
)

if wind_parquet.exists():
    frame = pd.read_parquet(wind_parquet)
    print(f"Loaded Week 3 parquet: {len(frame):,} rows")
else:
    frame = simulate_wind_forecast(WindSimulationConfig(n_days=120, seed=7))
    print(f"Using synthetic wind: {len(frame):,} hourly rows")

cols = feature_columns()
train, cal, test = chronological_conformal_split(frame, gap_hours=24)
print(f"Train={len(train):,}  Cal={len(cal):,}  Test={len(test):,}")

## 3. Fit quantile LightGBM + MAPIE CQR

Port of Week 3 `QuantileGBM` (LightGBM only) with locked hyperparameters from `final_model_params.json`.

In [ ]:
model = QuantileLGBM(model_params=WEEK3_MODEL_PARAMS)
result = run_conformal_cqr(train, cal, test, cols, model=model)

## 4. First coverage numbers

Compare **raw quantile-model intervals** vs **conformalized (CQR)** on the held-out test block.

In [ ]:
rows = [
    {
        "interval": "Raw P10–P90",
        "nominal": "80%",
        "coverage": result.raw_80.coverage,
        "gap": result.raw_80.coverage_gap,
        "mean_width_mw": result.raw_80.mean_width,
    },
    {
        "interval": "CQR 80%",
        "nominal": "80%",
        "coverage": result.cqr_80.coverage,
        "gap": result.cqr_80.coverage_gap,
        "mean_width_mw": result.cqr_80.mean_width,
    },
    {
        "interval": "Raw P05–P95",
        "nominal": "90%",
        "coverage": result.raw_90.coverage,
        "gap": result.raw_90.coverage_gap,
        "mean_width_mw": result.raw_90.mean_width,
    },
    {
        "interval": "CQR 90%",
        "nominal": "90%",
        "coverage": result.cqr_90.coverage,
        "gap": result.cqr_90.coverage_gap,
        "mean_width_mw": result.cqr_90.mean_width,
    },
]
coverage_table = pd.DataFrame(rows)
coverage_table

**Typical finding:** raw quantiles **under-cover**; CQR hits nominal (± finite-sample noise) by **widening** intervals.

In [ ]:
plot_df = result.test_frame.sort_values("valid_time").head(168)
t = plot_df["valid_time"]
y = plot_df["wind_mw"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(t, plot_df["raw_p10"], plot_df["raw_p90"], alpha=0.25, label="Raw P10–P90")
ax.fill_between(t, plot_df["cqr80_lo"], plot_df["cqr80_hi"], alpha=0.25, label="CQR 80%")
ax.plot(t, y, color="black", linewidth=0.8, label="Actual")
ax.plot(t, plot_df["pred_p50"], color="C0", linewidth=0.8, label="P50")
ax.set_ylabel("Wind (MW)")
ax.set_title("One week: raw vs conformalized 80% envelope")
ax.legend(loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Dispatch read

| Finding | Operational impact |
|---|---|
| Raw P10–P90 under-covers | Reserve buffer sized from the label is **too small** — more balancing activation and imbalance exposure |
| CQR hits ~80% / ~90% | Bands are **honest** for the calibration window — wider, but trustworthy |
| Wider CQR intervals | Dispatchers hold **more flex** around P50 in risky hours |

Week 3 pinball tuning optimizes **quantile loss**, not **coverage**. CQR is the distribution-free layer that closes the gap.

## 6. What would break

- **One test slice** — not a multi-year rolling backtest; coverage can drift by season.
- **Exchangeability** — regime shifts (fronts, fleet changes) violate the fine print; use rolling calibration or EnbPI next.
- **Feature pipeline drift** — CQR corrects **marginal** coverage, not bias from stale NWP or broken lags.
- **Synthetic data** — numbers here illustrate the method; re-run on real Week 3 parquet for production claims.